In [3]:
import time
import requests
import pandas as pd

from config import API_KEY

# Endpoint base da Alpha Vantage
BASE_URL = "https://www.alphavantage.co/query"


def cotacao_acao(symbol):
    """
    Consulta cotação de ações/ETFs via Alpha Vantage
    usando a função GLOBAL_QUOTE.
    """

    params = {
        "function": "GLOBAL_QUOTE",  # endpoint de cotação atual
        "symbol": symbol,            # ticker da ação
        "apikey": API_KEY            # chave da API
    }

    # Faz a requisição HTTP
    r = requests.get(BASE_URL, params=params, timeout=30)

    # Converte resposta JSON
    dados = r.json()

    # Extrai bloco principal da resposta
    quote = dados.get("Global Quote", {})

    # Normaliza retorno para estrutura padrão
    return {
        "Ativo": symbol,
        "Preço": quote.get("05. price"),
        "Variação": quote.get("09. change"),
        "Variação %": quote.get("10. change percent")
    }


def cotacao_crypto(symbol):
    """
    Consulta cotação de criptomoedas via Alpha Vantage
    usando taxa de câmbio BTC/USDT, ETH/USDT etc.
    """

    params = {
        "function": "CURRENCY_EXCHANGE_RATE",
        "from_currency": symbol,
        "to_currency": "USD",
        "apikey": API_KEY
    }

    r = requests.get(BASE_URL, params=params, timeout=30)
    dados = r.json()

    # Estrutura específica de resposta de cripto
    exchange = dados.get("Realtime Currency Exchange Rate", {})

    return {
        "Ativo": symbol,
        "Preço": exchange.get("5. Exchange Rate"),
        "Variação": "",
        "Variação %": ""
    }


# Lê a carteira de ativos definida pelo usuário
carteira = pd.read_csv("carteira.csv")

# Lista final de resultados normalizados
resultado = []


# Itera sobre todos os ativos da carteira
for _, linha in carteira.iterrows():

    ativo = linha["Ativo"]
    tipo = linha["Tipo"]

    try:
        # Decide qual função usar baseado no tipo do ativo
        if tipo == "STOCK":
            resultado.append(cotacao_acao(ativo))
        else:
            resultado.append(cotacao_crypto(ativo))

        print(f"{ativo} OK")

    except Exception as e:
        # Evita quebra do pipeline caso um ativo falhe
        print(f"{ativo}: {e}")

    # Respeita limite da API gratuita (5 chamadas/minuto)
    time.sleep(12)


# Converte lista de dicionários em DataFrame tabular
df = pd.DataFrame(resultado)

# Exporta resultado para consumo no Excel / Power BI
df.to_csv("cotacoes.csv", index=False)

print("\nArquivo cotacoes.csv gerado com sucesso!")

BTC OK
SHIB OK
GALA OK
VANRY OK
JASMY OK
COTI OK
IOST OK
WIN OK
SANTOS OK
BNB OK
PERL OK
ADA OK
SSV OK
DOGE OK
VOO OK
MSFT OK
NVDA OK
GOOGL OK
LLY OK
PLTR OK
JNJ OK
KO OK
AMD OK
AAPL OK
TSLA OK

Arquivo cotacoes.csv gerado com sucesso!


In [7]:
import csv
import requests

BASE_URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{}/dados/ultimos/1?formato=json"

INDICADORES = {
    "CDI Atual (%)": 12,
    "IPCA Atual (%)": 13522,
    "Selic Atual (%)": 432,
}

USD_URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados/ultimos/1?formato=json"

# Cria e abre o arquivo CSV para escrita
with open("indicadores.csv", mode="w", newline="", encoding="utf-8") as arquivo:
    escritor = csv.writer(arquivo)

    # Cabeçalho
    escritor.writerow(["Indicador", "Valor (%)"])

    # Indicadores econômicos
    for nome, codigo in INDICADORES.items():
        url = BASE_URL.format(codigo)
        r = requests.get(url)
        r.raise_for_status()

        valor = float(r.json()[0]["valor"].replace(",", "."))

        escritor.writerow([nome, f"{valor:.2f}"])
        print(f"{nome}: {valor:.2f}%")

    # ---------------------------
    # USD/BRL (câmbio dólar)
    # ---------------------------
    r = requests.get(USD_URL)
    r.raise_for_status()

    usd_brl = float(r.json()[0]["valor"].replace(",", "."))

    escritor.writerow(["USD/BRL (Dólar)", f"{usd_brl:.4f}"])
    print(f"USD/BRL (Dólar): {usd_brl:.4f}")

print("\nArquivo 'indicadores.csv' salvo com sucesso!")

CDI Atual (%): 0.05%
IPCA Atual (%): 4.72%
Selic Atual (%): 14.25%
USD/BRL (Dólar): 5.1695

Arquivo 'indicadores.csv' salvo com sucesso!
